In [0]:
-- ============================================================================
-- Column Mask for total_spent
-- ============================================================================

-- Step 1: Create governed tag (do this via UI first)
--   1. Go to Catalog > Governed Tags
--   2. Click "Create governed tag"
--   3. Tag key: financial_sensitivity
--   4. Allowed values: high, medium, low

-- Step 2: Create masking function
CREATE OR REPLACE FUNCTION sales.default.mask_financial_amount(amount DOUBLE)
RETURNS DOUBLE
RETURN -1.0;

-- Step 3: Create ABAC policy
CREATE OR REPLACE POLICY sales_financial_mask
ON CATALOG sales
COMMENT 'Mask high-sensitivity financial columns for non-finance users'
COLUMN MASK sales.default.mask_financial_amount
TO `account users`
EXCEPT `finance_team`
FOR TABLES
MATCH COLUMNS has_tag_value('financial_sensitivity', 'high') AS amount
ON COLUMN amount;

-- Step 4: Apply the tag
ALTER TABLE sales.gold.sales_fact
  ALTER COLUMN total_spent
  SET TAGS ('financial_sensitivity' = 'high');

-- ============================================================================
-- Verification
-- ============================================================================

SHOW POLICIES ON CATALOG sales;

DESCRIBE POLICY sales_financial_mask ON CATALOG sales;

SHOW EFFECTIVE POLICIES ON TABLE sales.gold.sales_fact;

-- Check tag is applied
SELECT * 
FROM sales.information_schema.column_tags 
WHERE table_name = 'sales_fact' 
  AND column_name = 'total_spent';

-- Test the query - finance_team should see actual values
SELECT 
  transaction_id,
  customer_id,
  total_spent,
  transaction_date
FROM sales.gold.sales_fact
LIMIT 10;